# Tutorial 09: AutoImprove, letting agents improve your pipeline

You have a working model. Is it the best you can do with this data? `pyhealth.autoimprove` searches for a better pipeline for you: a few agents take turns proposing a model and its settings, each proposal is trained and scored, and the agents learn from each other's results. Proposals come from an LLM (Claude, GPT or Gemini) or from random search, which needs no API key.

A search that tries many configs on the same data can fool you: the top score is partly luck. So AutoImprove is built to give you a result you can defend:

- every trial is scored by **patient-grouped cross-validation** and compared with your starting pipeline **fold by fold**;
- among the configs that really beat it, the **simplest one within noise** is selected (the one-SE rule);
- the test split is used **once**, at the end, for a paired comparison with confidence intervals;
- a **report** explains every trial: what was proposed, why, what changed and what it bought.

**What you will learn**

1. How to run a search from a starting pipeline, with random search or an LLM.
2. How to read the leaderboard and why the selected config is not always the top score.
3. How to measure what each changed setting contributed.
4. How to test the selection once and read the report.
5. How a null-label control catches a search that "improves" on noise.

**Time:** about 5 minutes on a free Colab CPU runtime. **Data:** a small synthetic cohort built in the notebook, so nothing to download and no credentials.

**How to run:** Runtime > Run all, or Shift+Enter cell by cell. The text above each code cell says what it does; the text below says how to read its output. The numbers quoted in the text come from the saved run; yours may differ slightly on other hardware.

## Setup

AutoImprove is newer than the PyHealth 2.1.0 release on PyPI, so this installs PyHealth from GitHub. If the install upgrades a package Colab already loaded, the runtime restarts once ("Your session crashed"); choose **Runtime > Run all** again.

In [1]:
# Installs PyHealth from GitHub (about 1-2 minutes the first time). If it
# upgrades a package Python has already loaded (Colab preloads a few, such as
# numpy), the runtime restarts once: Colab then shows "Your session crashed".
# That is expected. Choose Runtime > Run all again; this cell then skips the
# install.
import importlib.metadata as md
import importlib.util
import os
import subprocess
import sys

PYHEALTH_SOURCE = "pyhealth @ git+https://github.com/sunlabuiuc/PyHealth.git"


def installed(dist):
    try:
        return md.version(dist)
    except md.PackageNotFoundError:
        return None


if importlib.util.find_spec("pyhealth.autoimprove") is None:
    owners = md.packages_distributions()
    loaded = {}
    for module in list(sys.modules):
        for dist in owners.get(module, []):
            loaded[dist] = installed(dist)

    print("Installing PyHealth from GitHub...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", PYHEALTH_SOURCE],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(result.stdout[-5000:], result.stderr[-5000:])
        raise RuntimeError("Installing PyHealth failed; see the log above.")

    changed = sorted(d for d, v in loaded.items() if installed(d) != v)
    if changed:
        print("Upgraded packages that were already loaded:", ", ".join(changed))
        print("Restarting the runtime. When Colab says the session crashed, "
              "choose Runtime > Run all again.")
        os.kill(os.getpid(), 9)  # Colab restarts the runtime automatically

import logging
import warnings

import torch

# Keep the output readable: show one line per search cycle, hide the logs of
# the many individual trainings.
logging.basicConfig(format="%(message)s")
logging.getLogger("pyhealth").setLevel(logging.WARNING)
logging.getLogger("pyhealth.autoimprove").setLevel(logging.INFO)
warnings.filterwarnings("ignore")
print(f"PyHealth {installed('pyhealth')} (with AutoImprove) is ready | device: "
      f"{'GPU' if torch.cuda.is_available() else 'CPU'}")

PyHealth 2.1.0 (with AutoImprove) is ready | device: CPU


## 1. A cohort where we know the answer

To judge a search you need to know what it should find. So we simulate 2,100 patients, each with 3 to 9 ICD-9 diagnosis codes, and a 30-day readmission label whose risk comes from four codes only: heart failure (428.0), acute kidney injury (584.9), diabetes (250.00) and hypertension (401.9). Each adds a fixed amount to the log-odds; the other 26 codes add nothing. About 21% of patients are readmitted.

The patients are split 1,200 / 300 / 600 into training, validation and test. Processors (the code vocabulary) are fitted on the training patients only. With real data you would write `train, val, test = dataset.set_task(task, split=PatientSplit((0.6, 0.15, 0.25)))` instead; everything after this cell stays the same.

In [2]:
import math
import random

from pyhealth.datasets import create_sample_dataset

RISK = {"428.0": 1.8, "584.9": 1.3, "250.00": 0.5, "401.9": 0.2}  # log-odds
OTHER = ["414.01", "272.4", "427.31", "530.81", "244.9", "285.9", "496",
         "599.0", "276.1", "V45.81", "403.90", "486", "038.9", "518.81",
         "995.92", "305.1", "311", "V58.61", "424.1", "433.10", "562.10",
         "787.91", "780.39", "707.03", "300.00", "V15.82"]
CODES = list(RISK) + OTHER


def simulate(n, seed, first_id=0):
    """n patients: diagnosis codes and a readmission label drawn from RISK."""
    rng = random.Random(seed)
    patients = []
    for i in range(n):
        codes = rng.sample(CODES, rng.randint(3, 9))
        logit = -2.4 + sum(RISK.get(c, 0.0) for c in codes)
        label = int(rng.random() < 1 / (1 + math.exp(-logit)))
        patients.append({"patient_id": f"p{first_id + i}",
                         "conditions": codes, "readmitted": label})
    return patients


schema = ({"conditions": "sequence"}, {"readmitted": "binary"})
train = create_sample_dataset(simulate(1200, seed=0), *schema)
fitted = {"input_processors": train.input_processors,
          "output_processors": train.output_processors}
val = create_sample_dataset(simulate(300, seed=1, first_id=10_000), *schema, **fitted)
test = create_sample_dataset(simulate(600, seed=2, first_id=20_000), *schema, **fitted)

for name, ds in [("train", train), ("validation", val), ("test", test)]:
    labels = [int(s["readmitted"]) for s in ds]
    print(f"{name:10s} {len(labels):5d} patients, {sum(labels) / len(labels):.0%} readmitted")

train       1200 patients, 21% readmitted
validation   300 patients, 20% readmitted
test         600 patients, 21% readmitted


## 2. The starting pipeline and the search space

The **baseline** is the pipeline you have today: here a GRU trained for 3 epochs, a typical first attempt. Every trial is compared with it.

The **search space** lists the models and argument ranges agents may propose; anything outside it is rejected before training. The default space has six sequence models. To keep this run short we allow three, one per rung of the *interpretability ladder*: logistic regression (rung 1, each code has one odds ratio), an MLP (rung 3) and an RNN (rung 4).

In [3]:
from pyhealth.autoimprove import (DEFAULT_MODELS, LADDER, PipelineConfig,
                                  SearchSpace)

baseline = PipelineConfig("RNN", {"hidden_dim": 64}, {"epochs": 3})
space = SearchSpace({name: DEFAULT_MODELS[name]
                     for name in ("LogisticRegression", "MLP", "RNN")})

for name in space.models:
    rung = space.rung(PipelineConfig(name))
    print(f"{name:20s} rung {rung}: {LADDER[rung]}")
print("\nRNN arguments the agents may set:", ", ".join(space.models["RNN"].params))
print("Training settings:", ", ".join(space.train_params))

LogisticRegression   rung 1: linear: each input has one coefficient (an odds ratio)
MLP                  rung 3: small interactions between inputs
RNN                  rung 4: deep interactions or sequence models: explained only after the fact

RNN arguments the agents may set: embedding_dim, hidden_dim, rnn_type, num_layers, dropout, bidirectional
Training settings: epochs, lr, weight_decay, batch_size, patience


## 3. Who proposes: random search, your own ideas, or an LLM

A *proposer* returns the next config to try. Proposals are **data, never code**: a model name and settings, checked against the space. Nothing a proposer returns is executed.

- `RandomProposer` samples the space. It needs no API key and is the baseline any smarter proposer should beat.
- `LLMProposer(ClaudeLLM())` asks an LLM, which sees the leaderboard, each agent's history and the lessons learned so far (section 8).
- Any class with a `propose(context)` method works. Below, a proposer tries one idea of ours first, then falls back to random search. The `rationale` and `hypothesis` are written **before** the trial runs and end up in the report, so you can later check whether the idea paid off.

In [4]:
from pyhealth.autoimprove import Proposer, RandomProposer


class OurIdeasFirst(Proposer):
    """Tries hand-written configs first, then random search."""

    def __init__(self, ideas, seed=0):
        self.ideas = list(ideas)
        self.random = RandomProposer(seed)

    def propose(self, context):
        return self.ideas.pop(0) if self.ideas else self.random.propose(context)


simplest = PipelineConfig(
    "LogisticRegression", {}, {"epochs": 30, "lr": 0.01},
    rationale="Each diagnosis may add a fixed amount of risk, so try the "
              "simplest model first.",
    hypothesis={"expected": "PR-AUC close to the best neural model",
                "kind": "clinical"},
)
proposer = OurIdeasFirst([simplest])

## 4. Run the search

Three agents propose once per cycle, for three cycles, so 9 trials plus the baseline. Each cycle the agents follow a strategy in turn (explore, refine, crossover, contrast), as in Google's [codegen-classroom](https://github.com/google-research/codegen-classroom).

`cv_folds=3` scores every trial by 3-fold cross-validation over the training patients, grouped by patient, so every trial is compared with the baseline on the same three folds. The validation split is then used only for early stopping. The metric defaults to PR-AUC for a binary label. `run_dir` saves every trial; run the cell again with the same directory and the search resumes instead of starting over. About a minute on a CPU.

In [5]:
import shutil

from pyhealth.autoimprove import AutoImprover

shutil.rmtree("runs/readmission", ignore_errors=True)  # start fresh
improver = AutoImprover(
    train, val, proposer,
    baseline=baseline, space=space,
    n_agents=3, n_cycles=3,
    cv_folds=3,
    run_dir="runs/readmission",
    seed=0,
)
result = improver.run()

cycle 0: selected #3, pr_auc=0.3797138189246501
cycle 1: selected #3, pr_auc=0.3797138189246501
cycle 2: selected #3, pr_auc=0.3797138189246501


## 5. Read the leaderboard

The leaderboard ranks finished trials by their mean score over the folds. The selected trial is marked.

In [6]:
import pandas as pd

rows = []
for t in result.leaderboard(k=10):
    rows.append({
        "trial": t.trial_id,
        "agent": t.agent,
        "model": t.config.model,
        "PR-AUC (CV)": round(t.score, 3),
        "selected": "yes" if t is result.best else "",
        "settings": {k: (round(v, 5) if isinstance(v, float) else v)
                     for k, v in {**t.config.model_params,
                                  **t.config.train_params}.items()},
    })
pd.set_option("display.max_colwidth", 120)
display(pd.DataFrame(rows).set_index("trial"))
print(improver.selection_rule())

,agent,model,PR-AUC (CV),selected,settings
trial,,,,,
3,agent2,MLP,0.380,yes,"{'embedding_dim': 107, 'hidden_dim': 165, 'n_layers': 2, 'epochs': 33, 'lr': 4e-05, 'weight_decay': 0.0, 'batch_size': 16, 'patience': 10}"
7,agent0,RNN,0.370,,"{'embedding_dim': 238, 'hidden_dim': 32, 'rnn_type': 'GRU', 'num_layers': 3, 'dropout': 0.15519, 'bidirectional': False, 'epochs': 35, 'lr': 0.00062, 'weight_decay': 0.0, 'batch_size': 64, 'patience': 9}"
6,agent2,RNN,0.370,,"{'embedding_dim': 250, 'hidden_dim': 19, 'rnn_type': 'GRU', 'num_layers': 3, 'dropout': 0.58806, 'bidirectional': False, 'epochs': 40, 'lr': 0.00094, 'weight_decay': 0.02178, 'batch_size': 32, 'patience': 6}"
1,agent0,LogisticRegression,0.346,,"{'embedding_dim': 128, 'epochs': 30, 'lr': 0.01, 'weight_decay': 0.0, 'batch_size': 64, 'patience': 3}"
4,agent0,MLP,0.334,,"{'embedding_dim': 248, 'hidden_dim': 152, 'n_layers': 2, 'epochs': 20, 'lr': 2e-05, 'weight_decay': 0.0, 'batch_size': 64, 'patience': 8}"
9,agent2,RNN,0.288,,"{'embedding_dim': 156, 'hidden_dim': 166, 'rnn_type': 'LSTM', 'num_layers': 2, 'dropout': 0.06414, 'bidirectional': True, 'epochs': 21, 'lr': 0.00201, 'weight_decay': 3e-05, 'batch_size': 32, 'patience': 3}"
2,agent1,MLP,0.273,,"{'embedding_dim': 210, 'hidden_dim': 243, 'n_layers': 4, 'epochs': 3, 'lr': 0.00011, 'weight_decay': 0.00036, 'batch_size': 128, 'patience': 5}"
0,baseline,RNN,0.250,,"{'hidden_dim': 64, 'embedding_dim': 128, 'rnn_type': 'GRU', 'num_layers': 1, 'dropout': 0.5, 'bidirectional': False, 'epochs': 3, 'lr': 0.001, 'weight_decay': 0.0, 'batch_size': 64, 'patience': 3}"
8,agent1,MLP,0.227,,"{'embedding_dim': 43, 'hidden_dim': 93, 'n_layers': 3, 'epochs': 46, 'lr': 3e-05, 'weight_decay': 5e-05, 'batch_size': 256, 'patience': 4}"


Every trial is compared with the starting pipeline fold by fold. A trial is a candidate if its gain is above 0 by more than 1 standard error(s), so a gain must be consistent across folds, not only high on average. Among the candidates, the selection is the most interpretable model (lowest rung) whose score is within one paired standard error of the best candidate's: simpler models win ties.


**Reading it:** the selected MLP lifts cross-validated PR-AUC from 0.25 to 0.38. Several trials score about the same (two RNNs at 0.37), while others are worse than the baseline: random search wastes many trials, which is why a guided proposer helps.

The selection rule is what makes the result trustworthy. A trial becomes a *candidate* only if its gain over the baseline is positive by more than one standard error **across folds**, so a gain carried by one lucky fold does not count. Among candidates, the most interpretable model within one standard error of the best is selected. Our logistic-regression idea (trial 1) was a candidate, but it scored 0.346, and its gap to the MLP was larger than the noise, so the MLP was selected; had the two been within noise, logistic regression would have won the tie. Our hypothesis ("close to the best neural model") is now checked against a measurement, and it did not hold for this one setting of the learning rate. The true risk here *is* additive, so a few more trials tuning logistic regression might close the gap: a search is only as good as what it tries.

## 6. What did each change contribute?

`attribute()` reverts each setting that differs from the reference, one at a time, and re-scores on the same folds. A setting's contribution is the drop in score when it is reverted, with its paired standard error. The selected model differs from the baseline's, so the reference is the MLP's own default settings.

In [7]:
attribution = improver.attribute()
print("Reference:", attribution["reference"])
display(pd.DataFrame([{
    "setting": r["setting"],
    "from": f'{r["from"]:.3g}' if isinstance(r["from"], float) else r["from"],
    "to": f'{r["to"]:.3g}' if isinstance(r["to"], float) else r["to"],
    "contribution": round(r["contribution"], 3),
    "± SE": round(r["se"], 3),
    "folds better": f'{r["folds_better"]}/3',
} for r in attribution["rows"]]).set_index("setting"))

Reference: the default settings of MLP


,from,to,contribution,± SE,folds better
setting,,,,,
embedding_dim,128,107,0.004,0.028,2/3
hidden_dim,128,165,-0.001,0.005,1/3
batch_size,64,16,0.001,0.002,1/3
epochs,10,33,0.001,0.004,2/3
lr,0.001,3.61e-05,0.010,0.012,2/3
patience,3,10,0.000,0.004,1/3
weight_decay,1e-06,5e-06,-0.000,0.000,0/3


**Reading it:** every changed setting contributes about zero, within about one standard error. So the MLP's own defaults would score about as well: the gain over the baseline comes from switching to an MLP trained to convergence, not from the fine-tuned values. That is typical: a search changes many settings at once and only a few matter. Before you adopt a config, keep the settings that measurably help and treat the rest as noise.

## 7. Test once

Only now is the test split used, once. `evaluate_on_test` retrains the selected and the baseline configs and scores both on the 600 test patients, with patient-level bootstrap intervals and a **paired** interval for the difference, plus calibration and what flagging the top 10% of risk would catch.

In [8]:
test_result = improver.evaluate_on_test(test, n_boot=500)
for name in ("baseline", "best"):
    s = test_result[name]
    top10 = s["clinical"]["alerts"]["top 10%"]
    print(f"{name:9s} PR-AUC {s['estimate']:.3f} [{s['lower']:.3f}, {s['upper']:.3f}]"
          f" | top 10% flagged: sensitivity {top10.get('sensitivity', float('nan')):.2f},"
          f" PPV {top10.get('ppv', float('nan')):.2f}")
d = test_result["difference"]
print(f"selected - baseline: {d['estimate']:+.3f} [{d['lower']:+.3f}, {d['upper']:+.3f}]")

baseline  PR-AUC 0.305 [0.238, 0.396] | top 10% flagged: sensitivity 0.20, PPV 0.42
best      PR-AUC 0.439 [0.356, 0.533] | top 10% flagged: sensitivity 0.23, PPV 0.47
selected - baseline: +0.134 [+0.052, +0.205]


**Reading it:** the gain holds on patients the search never saw, and the paired 95% interval for the difference excludes zero. The selected model's calibration and top-10% alert performance are recorded too; the report below flags it if the probabilities need recalibrating before they are read as risks.

## The report

`report()` writes `report.html`, `report.md` and `report.json` to the run directory: the selected model in plain words, why it was selected, what each setting contributed, the test comparison, warnings (*flags*) that need a reviewer, and the reasoning trace of every trial. Statements are labelled by their evidence: measured, tested or hypothesis. It holds configs and aggregate scores only, never patient records. Here is the start of the Markdown version.

In [9]:
report = improver.report()
text = open("runs/readmission/report.md").read()
print(text[:2500], "\n...")
print("\nFlags for a reviewer:")
for flag in improver.flags():
    print(f"- {flag['id']}: {flag['detail']}")

# AutoImprove report

## Summary

**Selected: #3, MLP (a feed-forward neural network: inputs are combined in hidden layers, so their effects can interact).** CV pr_auc 0.380 against 0.250 for the starting pipeline: +0.130 ± 0.022 (better in 3/3 folds). *(measured)*

On the test patients, pr_auc was 0.439 [0.356, 0.533] against 0.305 [0.238, 0.396] for the starting pipeline; difference +0.134 [+0.052, +0.205]. *(measured)*

Flagging the top 10% of test samples by risk would catch 23% of events, with 47% of flags being events; observed / expected events 1.06, calibration slope 1.44. *(measured)*

**Why this model.** It beat the starting pipeline consistently across folds and has the highest score of the 8 candidates (rung 3: small interactions between inputs). No simpler candidate scored within one standard error of it. *(measured)*

**Clinical review.** 2 item(s) flagged for a clinician, 2 still open: see *Clinician review*.

## Task and data

No task card was given, so the cohort, outc

In Colab, open `runs/readmission/report.html` from the file browser on the left for the full report. Flags such as `no_task_card` (no clinical rationale recorded for the cohort and features) are meant to be reviewed and signed off with `improver.approve(...)`, so a report that goes to a clinician shows what was checked.

## 8. A sanity check: does the search find gains in pure noise?

A search that always "finds an improvement" is worthless. So repeat it with the label replaced by a coin flip at the same rate: no model can predict it, and an honest search should report no gain. Two guards apply. The candidate rule above only selects a trial whose gain over the baseline is consistent across folds. And a *signal check* compares the baseline and the selection with a chance-level ranking on the same folds, raising a `no_signal` flag when either does not beat chance (for example a weak baseline that scores *below* chance, so that anything reaching chance looks like a gain). The check is saved to `signal.json` in the run directory.

In [10]:
rng = random.Random(42)
null_train = simulate(1200, seed=0)
for p in null_train:
    p["readmitted"] = int(rng.random() < 0.21)  # unrelated to the codes
null_train = create_sample_dataset(null_train, *schema, **fitted)

import json

shutil.rmtree("runs/null_label", ignore_errors=True)
null_improver = AutoImprover(
    null_train, val, RandomProposer(seed=0),
    baseline=baseline, space=space,
    n_agents=3, n_cycles=2, cv_folds=3, run_dir="runs/null_label", seed=0,
)
null_result = null_improver.run()
print("Selected:", "the baseline (no improvement found)"
      if null_result.best is null_result.baseline else f"trial {null_result.best.trial_id}")

for run in ("readmission", "null_label"):
    check = json.load(open(f"runs/{run}/signal.json"))
    chance = sum(check["chance"]) / len(check["chance"])
    sel = check["selected"]
    print(f"{run:12s} chance PR-AUC {chance:.3f} | selected minus chance "
          f"{sel['gain']:+.3f} ± {sel['se']:.3f} | no_signal: {check['no_signal']}")

cycle 0: selected #0, pr_auc=0.23414229698533495
cycle 1: selected #0, pr_auc=0.23414229698533495
Selected: the baseline (no improvement found)
readmission  chance PR-AUC 0.219 | selected minus chance +0.161 ± 0.030 | no_signal: False
null_label   chance PR-AUC 0.207 | selected minus chance +0.027 ± 0.012 | no_signal: False


**Reading it:** on the coin-flip label no trial beat the baseline consistently, so the baseline stays selected and the search reports no improvement, as it should. The signal check puts numbers on it: on the real label the selection beats chance by about 0.16, on the coin flip by about 0.03, which is luck on three small folds (just enough to avoid the flag here). On your own data, run this control once: if your real search's gain over chance is not clearly larger than the null run's, it is not a gain.

## 9. Using an LLM as the proposer

With an LLM, agents reason about the results: they see the leaderboard, their own history and the strategy for the turn, state a hypothesis, and after each cycle the proposer rewrites a short list of **lessons**, which later prompts build on. Pass `lessons_path` to carry lessons across runs of the same kind of search. Prompts hold the search space, dataset sizes, aggregate label counts and scores, never patient records.

The cell below runs only if an API key is set (`pip install anthropic` and set `ANTHROPIC_API_KEY`; `OpenAILLM` and `GeminiLLM` work the same way, and any `llm(system, prompt) -> str` function does too).

In [11]:
if os.environ.get("ANTHROPIC_API_KEY"):
    from pyhealth.autoimprove import ClaudeLLM, LLMProposer

    shutil.rmtree("runs/readmission_llm", ignore_errors=True)
    llm_improver = AutoImprover(
        train, val, LLMProposer(ClaudeLLM()),
        baseline=baseline, space=space,
        n_agents=3, n_cycles=3, cv_folds=3,
        run_dir="runs/readmission_llm", lessons_path="runs/lessons.json", seed=0,
    )
    llm_result = llm_improver.run()
    print(llm_result.summary())
    print("Lessons:", *llm_result.lessons, sep="\n- ")
else:
    print("No ANTHROPIC_API_KEY set: skipped. Random search above needs no key.")

No ANTHROPIC_API_KEY set: skipped. Random search above needs no key.


## 10. On your own data

```python
from pyhealth.autoimprove import AutoImprover, ClaudeLLM, LLMProposer, PipelineConfig
from pyhealth.datasets import PatientSplit

train, val, test = dataset.set_task(task, split=PatientSplit((0.6, 0.15, 0.25)))
improver = AutoImprover(
    train, val, LLMProposer(ClaudeLLM()),        # or RandomProposer()
    baseline=PipelineConfig("RNN", {"hidden_dim": 128}, {"epochs": 10}),
    cv_folds=5, n_agents=3, n_cycles=5,
    run_dir="runs/my_task",                      # resumes if interrupted
    confirm=True,                                # re-score the top candidates on new folds
    task_card=task.task_card(),                  # clinical rationale for the report
)
result = improver.run()
improver.attribute()
improver.evaluate_on_test(test)                  # once
improver.report()
```

**Tips**

- Use `cv_folds=5` when you can. With few positive patients per fold, the improver warns (`few_events`) because one lucky fold can carry a whole gain.
- `confirm=True` re-scores the top candidates on new folds and extra seeds before the test, which removes part of the winner's curse.
- For tabular features, add XGBoost to the space with `with_xgboost(space)`; it sits on rung 2 of the ladder (additive effects).
- `features=FeatureSpace(...)` lets agents propose input features as well as models, each accepted only if its paired gain clears the bar.

**Learn more:** the [AutoImprove API docs](https://pyhealth.readthedocs.io/en/latest/api/autoimprove.html), and the example scripts `examples/autoimprove_synthetic_mimic3.py`, `examples/autoimprove_eicu_ihm48.py` and `examples/autoimprove_features_mimic3.py`.